# Knowledge graphs & embeddings (TransE) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(x): return 1/(1+np.exp(-x))
def softmax(x):
    x=np.asarray(x,dtype=float); e=np.exp(x-x.max()); return e/e.sum()
def draw_graph(A, pos=None, values=None, title=''):
    A=np.asarray(A); n=A.shape[0]
    if pos is None:
        ang=np.linspace(0,2*np.pi,n,endpoint=False); pos=np.c_[np.cos(ang),np.sin(ang)]
    plt.figure(figsize=(4,4))
    for i in range(n):
        for j in range(i+1,n):
            if A[i,j]!=0: plt.plot([pos[i,0],pos[j,0]],[pos[i,1],pos[j,1]],c='0.75',lw=1+2*abs(A[i,j]))
    c=values if values is not None else np.arange(n)
    plt.scatter(pos[:,0],pos[:,1],c=c,s=320,cmap='viridis',edgecolor='k',zorder=3)
    for i,(x,y) in enumerate(pos): plt.text(x,y,str(i),ha='center',va='center',color='white',weight='bold')
    plt.axis('off'); plt.title(title)
print('setup ok')

## TransE scores triples by asking whether head plus relation lands near tail
Knowledge graph embeddings represent typed facts as triples. TransE uses the translation h+r≈t; the examples compute a positive distance, a corrupted negative, margin loss, inverse relations, and candidate ranking.

In [ ]:
# 1) positive triple distance ||h+r-t||
h=np.array([1.,1.]); r=np.array([2.,0.]); t=np.array([3.,1.]); d=np.linalg.norm(h+r-t)
plt.figure(figsize=(4,4)); plt.arrow(*h,*r,head_width=.08,length_includes_head=True); plt.scatter([h[0],t[0]],[h[1],t[1]],s=120); plt.title(f'positive distance {d:.1f}')
assert d==0

In [ ]:
# 2) corrupted tail has larger distance
h=np.array([1.,1.]); r=np.array([2.,0.]); t_bad=np.array([2.,2.]); d=np.linalg.norm(h+r-t_bad)
plt.figure(figsize=(5,3)); plt.bar(['bad triple distance'],[d]); plt.title('corrupted triple')
assert abs(d-math.sqrt(2))<1e-9

In [ ]:
# 3) margin ranking loss
pos=0.0; neg=math.sqrt(2); gamma=1.0; loss=max(0,gamma+pos-neg)
plt.figure(figsize=(5,3)); plt.bar(['margin loss'],[loss]); plt.title('max(0, gamma + pos - neg)')
assert abs(loss-0.0)<1e-9

In [ ]:
# 4) inverse relation is the negative translation
r=np.array([2.,0.]); inv=-r
plt.figure(figsize=(5,3)); plt.bar(['r_x','r_y','inv_x','inv_y'],[r[0],r[1],inv[0],inv[1]]); plt.title('inverse relation vector')
assert np.allclose(inv,[-2,0])

In [ ]:
# 5) rank candidate tails by distance
h=np.array([1.,1.]); r=np.array([2.,0.]); C=np.array([[3.,1.],[2.,2.],[0.,0.]]); dist=np.linalg.norm(h+r-C,axis=1)
plt.figure(figsize=(5,3)); plt.bar(['true','bad1','bad2'],dist); plt.title('TransE candidate ranking')
assert int(np.argmin(dist))==0